# GFF Feature Breakdown — Ensembl vs CATThis notebook reads the per‑assembly outputs from the new `GFF_COMPARE` module and plots global distributions and summaries.

In [ ]:
import pandas as pd, numpy as npimport matplotlib.pyplot as plt, matplotlib.patches as mpatchesfrom pathlib import PathOUTPUT_DIR  = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results')QC_DIR      = OUTPUT_DIR / 'qc_metrics'FIG_DIR     = OUTPUT_DIR / 'figures'FIG_DIR.mkdir(parents=True, exist_ok=True)print('QC_DIR =', QC_DIR)

In [ ]:
# Load gff_compare outputsfeat_rows, gene_rows, tx_rows = [], [], []for p in QC_DIR.rglob('gff_compare/feature_counts.tsv'):    try:        feat_rows.append(pd.read_csv(p, sep='	'))    except Exception as e:        print('WARN feat:', p, e)for p in QC_DIR.rglob('gff_compare/gene_metrics.tsv'):    try:        gene_rows.append(pd.read_csv(p, sep='	'))    except Exception as e:        print('WARN gene:', p, e)for p in QC_DIR.rglob('gff_compare/tx_metrics.tsv'):    try:        tx_rows.append(pd.read_csv(p, sep='	'))    except Exception as e:        print('WARN tx:', p, e)feat = pd.concat(feat_rows, ignore_index=True) if feat_rows else pd.DataFrame()genes = pd.concat(gene_rows, ignore_index=True) if gene_rows else pd.DataFrame()trans = pd.concat(tx_rows, ignore_index=True) if tx_rows else pd.DataFrame()print('Loaded:', len(feat), 'feature rows;', len(genes), 'gene rows;', len(trans), 'tx rows')

In [ ]:
# Plot 1: per-assembly total features (violins)if feat.empty:    print('No feature counts')else:    fig, axes = plt.subplots(2, 2, figsize=(10, 6))    metrics = [('n_genes','Genes'),('n_transcripts','Transcripts'),('n_exon','Exons'),('n_cds','CDS')]    for ax, (col, label) in zip(axes.flat, metrics):        data = [feat[feat['source']==src][col].values for src in ['Ensembl','CAT']]        parts = ax.violinplot(data, positions=[1,2], showmedians=True)        colors = ['#1f77b4','#ff7f0e']        for pc, colr in zip(parts['bodies'], colors):            pc.set_facecolor(colr); pc.set_alpha(0.6)        ax.set_xticks([1,2]); ax.set_xticklabels(['Ensembl','CAT'])        ax.set_ylabel(label + ' per assembly')        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)    plt.tight_layout(); plt.show()

In [ ]:
# Plot 2: transcripts per gene distribution (median-of-medians) by biotypeif genes.empty:    print('No gene metrics')else:    genes['biotype_group'] = genes['biotype_group'].astype('category')    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)    for ax, src in zip(axes, ['Ensembl','CAT']):        per_asm = (genes[genes['source']==src]                    .groupby(['assembly_accession','biotype_group'])['n_transcripts']                    .describe(percentiles=[.05,.25,.5,.75,.95]).reset_index())        # Build stats per biotype        stats, labels = [], []        for b in genes['biotype_group'].cat.categories:            sub = per_asm[per_asm['biotype_group']==b]            if sub.empty:                continue            stats.append({'med': sub['50%'].median(), 'q1': sub['25%'].median(), 'q3': sub['75%'].median(),                          'whislo': sub['5%'].median(), 'whishi': sub['95%'].median(), 'fliers': []})            labels.append(b)        pos = np.arange(1, len(stats)+1)        bp = ax.bxp(stats, positions=pos, widths=0.55, patch_artist=True, showfliers=False, medianprops=dict(color='black', linewidth=2))        for patch in bp['boxes']:            patch.set_facecolor('#74a9cf'); patch.set_alpha(0.6)        ax.set_xticks(pos); ax.set_xticklabels(labels, rotation=25, ha='right')        ax.set_title(src, fontsize=10, fontweight='bold', loc='left')        ax.set_ylabel('Transcripts per gene')        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)    plt.tight_layout(); plt.show()

In [ ]:
# Plot 3: transcript structure metrics (n_exons, tx_len_bp, cds_len_bp)if trans.empty:    print('No transcript metrics')else:    # Summaries by source    def plot_violin(metric, ylabel):        fig, ax = plt.subplots(figsize=(6.4, 3.2))        data = [trans[trans['source']==src][metric].dropna().values for src in ['Ensembl','CAT']]        parts = ax.violinplot(data, positions=[1,2], showmedians=True)        for pc, colr in zip(parts['bodies'], ['#1f77b4','#ff7f0e']):            pc.set_facecolor(colr); pc.set_alpha(0.6)        ax.set_xticks([1,2]); ax.set_xticklabels(['Ensembl','CAT'])        ax.set_ylabel(ylabel); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)        plt.tight_layout(); plt.show()    plot_violin('n_exons', 'Exons per transcript')    plot_violin('tx_len_bp', 'Transcript length (bp)')    plot_violin('cds_len_bp', 'CDS length (bp)')